# 09 · SkewerJAX and fake_spectra: the results compared

fake_spectra is the external reference code for spectra from simulations. This notebook compares what the two codes
compute on the same TNG50-4 skewer: the gas along the skewer, the optical depth $\tau$ and, above all, the **transmitted
flux** $F = e^{-\tau}$, with the summaries derived from it (mean flux $\langle F\rangle$, max $\vert\Delta F\vert$, the P1D in two
$k$ bands). Speed and
memory are in notebook `10_comparison_fake_spectra_performance`.

`skewerjax.comparison` re-implements fake_spectra's scheme on SkewerJAX's absorbers and runs fake_spectra itself:
- **Matched mode:** `FAKE_SPECTRA_EXTRACTION` + `FAKE_SPECTRA_INTEGRATION` reproduce fake_spectra (top-hat spheres,
  its mass and b constants, its star-forming rule, 7-point chords, per-cell pixel average, linear detuning, `tautail`,
  one redshift with a linear Hubble flow).
- **Default:** SkewerJAX's choices. Every difference is one named switch, and the **effect chain** goes from one to
  the other one switch at a time.
- `FakeSpectraSource` gives fake_spectra's own spectra and per-cell inputs, **live** (fake_spectra installed; each run
  in its own process with `python -m skewerjax.comparison.driver full …`, cached) or from a **reference** HDF5 file.

The path: the flux of one skewer in both modes against fake_spectra; the gas before any integration; each switch
alone; their root causes; the effect chain that ranks them; a summary.

The full set of checks is `scripts/validation/check_fake_spectra.py`. Runtime: ~1 min (live).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
import tempfile
from skewerjax.core import LineBundle
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, ExtractionOptions, HI
from skewerjax.comparison import (FakeSpectraSource, SKEWER_SETS, FAKE_SPECTRA_EXTRACTION as EF, FAKE_SPECTRA_INTEGRATION as IF,
                                  DEFAULT_INTEGRATION, INTEGRATION_EFFECTS, EXTRACTION_EFFECTS, matched_tau, run_effects, effect_rows,
                                  spectral_metrics)
CACHE = tempfile.mkdtemp(prefix="fs_cache_")             # where live fake_spectra results go (or pass reference="file.h5")
from skewerjax.utils.paths import DATA_DIR as DATA      # folder with TNG50-4/: $SKEWERJAX_DATA (docs/installation.md)
FS = FakeSpectraSource(cache=CACHE, snap_base=f"{DATA}/TNG50-4", snapnum=25)
print("fake_spectra:", FS.mode)
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
lya = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
sk = SimulationSkewers.axis_aligned(reader, SKEWER_SETS["single"]["cofm"], SKEWER_SETS["single"]["axis"], 0.25)
v, dv = sk.spectral_grid(1.0)                            # fake_spectra's periodic pixel grid
fs = FS.run("single_matched", "single", products="tau,colden")
t_matched = matched_tau(sk, HI, lya, v, EF, IF)
t_default = sk.tau(HI, lya, v, velocity_period=True)      # SkewerJAX default (exact frame), periodic over its own span as fake_spectra
print({k: f"{x:.1e}" for k, x in spectral_metrics(t_matched, fs["tau"], dv).items()})

In [ ]:
plot_dag(('simulation', 'tau', 'fake_spectra',), external=True); plt.show()

## The differences, step by step

Every difference between the two codes is one named switch. They act at two levels: how the gas is **extracted**
from the cells (layers 3 and 5 of notebook 05), and how the optical-depth integral is **evaluated** (layer 6). The
list below is printed from the package (`comparison.EXTRACTION_EFFECTS`, `INTEGRATION_EFFECTS`), with where each
choice comes from:

| switch | fake_spectra | SkewerJAX default | where it comes from |
|---|---|---|---|
| T constants | its $m_p$, $k_B$ (`unitsystem.py`) | CODATA 2018 | `gas_physics.temperature`; $T$ $8.1\times10^{-6}$ lower |
| b constants | its constants in $b$ | CODATA, $m_u$ | `comparison.effects`; $b^2$ ratio `FAKE_SPECTRA_B2_RATIO` |
| mass constant | $n = \rho X/(A m_p)$ | $n = \rho X/(A m_u)$ | `absorption.h` L4 labels PROTONMASS "1 a.m.u."; 0.73 % |
| neutral fraction (H I) | `sf_neutral` rule: $\rho/m_p$ threshold | stored $x_{\rm HI}$, star-forming cells recomputed | `gas_properties.get_reproc_HI` L116–L146 |
| CLOUDY density (metals) | hden $= 0.774\,\rho/m_p$ | $n_{\rm H}$ | `convert_cloudy.py` L183 |
| CLOUDY interpolation | cubic spline | bilinear in $\log f$ | `CloudyTable.ion` |
| deposition | top-hat spheres of radius $V^{1/3}$ | Voronoi cells | `absorption.cpp:tophat_kern_frac` L112–L118 |
| tautail | stop at $\tau < 10^{-7}$ per cell | full profiles | `absorption.cpp` L264, `spectra.py` L145 |
| pixel average | average over the pixel if $b < 2\,dv$ | pixel centres | `singleabs.h:tau_kern_outer` L133–L155 |
| detuning | linear, $H(a, \Delta u/b)$ | exact, $\nu = (c/\lambda)\,\mathrm{expm1}(-\Delta v/c)$ | `singleabs.h` L182; first order in $\Delta v/c$ |
| 7/8 normalisation | 7 interior chord points of weight 1/8 | (the exact chord) | `singleabs.h` L100–L118: a top hat integrates 7/8 of the column |
| chord quadrature | 7 points per chord | exact pieces of the dual segments | `comparison.emulation.fake_chord_absorbers` |
| frame | one $a$, $u = aHs + v_{\rm pec}$ | $a(x)$ along the path, $v_\dagger$ | `spectra.py` L228 ($v_{\max} = aHL$) |

**How each is quantified.** The *chain* switches fake_spectra's options to SkewerJAX's one at a time (integration on
fake_spectra's extraction, then the extraction on SkewerJAX's integration): it telescopes exactly, so the steps add up
to the total difference. *Alone* applies each switch by itself to fake_spectra's options, which isolates it.

In [ ]:
from skewerjax.comparison import EXTRACTION_EFFECTS as _EE, INTEGRATION_EFFECTS as _IE
for e in _EE + _IE:
    print(f"{e.kind:11s} {e.label:22s} ions={e.ions:6s} {e.what}")

In [ ]:
F_fs, F_m, F_d = np.exp(-fs["tau"][0]), np.exp(-t_matched[0]), np.exp(-t_default[0])
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
axs[0].plot(v, F_fs, color=EC["fake_spectra"], **MS["reference"], label="fake_spectra")
axs[0].plot(v, F_m, color=EC["HI"], **MS["matched"], label="SkewerJAX, matched")
axs[0].plot(v, F_d, color=EC["HI"], **MS["default"], alpha=0.5, label="SkewerJAX, default")
axs[1].plot(v, F_m - F_fs, color=EC["HI"], **MS["matched"], label="matched − fake_spectra")
axs[1].plot(v, F_d - F_fs, color=EC["HI"], **MS["default"], alpha=0.5, label="default − fake_spectra")
axs[0].set(ylabel="F", xlim=(1500, 2500)); axs[1].set(xlabel="u [km/s]", ylabel=r"$\Delta F$", ylim=(-1, 1))
axs[1].set_yscale("symlog", linthresh=1e-5); axs[1].set_yticks([-1e-1, -1e-3, -1e-5, 0, 1e-5, 1e-3, 1e-1])
for ax in axs: ax.legend(fontsize=15)
add_panel_letters(axs); fig.suptitle("One TNG50-4 skewer: matched mode reproduces fake_spectra (|ΔF| ~ 10⁻⁵); the default differs", fontsize=21); plt.show()

### Step 1: the gas, before any integration

The column per position pixel isolates the extraction from the integration. fake_spectra's own `colden`, SkewerJAX's
top-hat deposition with fake_spectra's constants (matched extraction) and SkewerJAX's default Voronoi deposition, on
the same skewer: the matched columns agree to the float32 floor, and the Voronoi cells move the gas between pixels
without changing the total much.

In [ ]:
N_fs = np.asarray(fs["colden"][0])
N_th = sk.column_per_pixel(HI, EF, nbins=v.size)[0]
N_vo = sk.column_per_pixel(HI, ExtractionOptions(), nbins=v.size)[0]
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
axs[0].semilogy(v, N_fs, color=EC["fake_spectra"], **MS["reference"], label="fake_spectra colden")
axs[0].semilogy(v, N_th, color=EC["tophat"], **MS["matched"], label="SkewerJAX, top-hat (matched extraction)")
axs[0].semilogy(v, N_vo, color=EC["voronoi"], **MS["default"], alpha=0.6, label="SkewerJAX, Voronoi (default)")
has = N_fs > 0
axs[1].plot(v[has], N_th[has] / N_fs[has] - 1, color=EC["tophat"], **MS["matched"], label="top-hat / fake_spectra − 1")
axs[1].plot(v[has], N_vo[has] / N_fs[has] - 1, color=EC["voronoi"], **MS["default"], alpha=0.6, label="Voronoi / fake_spectra − 1")
axs[1].set_yscale("symlog", linthresh=1e-6); axs[1].set_yticks([-1, -1e-3, -1e-6, 0, 1e-6, 1e-3, 1]); axs[0].set(ylabel=r"$N_{\rm HI}$ per pixel [cm$^{-2}$]", xlim=(1500, 2500))
axs[1].set(xlabel="pixel position [km/s of Hubble flow]", ylabel="relative difference")
for ax in axs: ax.legend(fontsize=15)
add_panel_letters(axs); fig.suptitle("The extraction alone: H I column per pixel on the skewer of the figure above", fontsize=21); plt.show()
print(f"total N_HI: fake_spectra {N_fs.sum():.4e} | top-hat {N_th.sum():.4e} ({N_th.sum() / N_fs.sum() - 1:+.1e}) | "
      f"Voronoi {N_vo.sum():.4e} ({N_vo.sum() / N_fs.sum() - 1:+.1e}); median per-pixel |top-hat / fs - 1| = {np.median(np.abs(N_th[has] / N_fs[has] - 1)):.1e}")

### One difference at a time: what each switch does to the flux

Every panel applies **one** switch alone to fake_spectra's options (`run_effects(...)["isolated"]`) and compares the
flux with fake_spectra's options (thick, light) on 1000 km/s of the skewer above; below, the difference. The
integration switches act on fake_spectra's extraction, the extraction switches on SkewerJAX's integration.

In [ ]:
from skewerjax.utils.plotting import plot_effect_fluxes
res_i = run_effects(sk, HI, lya, v, INTEGRATION_EFFECTS, EF, IF, verbose=False)
res_e = run_effects(sk, HI, lya, v, EXTRACTION_EFFECTS, EF, DEFAULT_INTEGRATION, verbose=False)
base_i = matched_tau(sk, HI, lya, v, EF, IF)
base_e = matched_tau(sk, HI, lya, v, EF, DEFAULT_INTEGRATION)
for res, base, what in ((res_i, base_i, "integration switches (on fake_spectra's extraction)"),
                        (res_e, base_e, "extraction switches (on SkewerJAX's integration)")):
    fig, axs = plot_effect_fluxes(v, base, res["isolated"], window=(1500, 2500), ncols=4)
    fig.suptitle(f"One switch at a time, H I Lyα: the {what}", fontsize=21); plt.show()

### Root causes, and why SkewerJAX's choice is the more accurate one

| switch | what fake_spectra does | root cause of the difference | why SkewerJAX's choice is more accurate |
|---|---|---|---|
| **7/8 normalisation** | a top hat sampled at 7 interior points of weight 1/8 | the 7-point rule of an SPH kernel applied to a top hat integrates 7/8 of the column | the column of a cell is a property of the gas: the exact piece lengths conserve it for any sampling |
| **chord quadrature** | 7 points per chord of every sphere | a fixed-order quadrature of a step function | the Voronoi pieces are the exact intersections of the path with the cells: no quadrature of the density |
| **deposition** | each cell as a sphere of radius $V^{1/3}$ of uniform density $3n/4\pi$ | spheres overlap and leave gaps, so the gas along the line is multi-valued and moves between pixels | Arepo's hydrodynamics is solved on the Voronoi cells; nearest-generator deposition reproduces the simulated fluid exactly (piecewise constant, single valued) |
| **frame** | one scale factor, linear Hubble flow $u = aHs$ across the box | $a(x)$ is frozen at the snapshot over the whole path | $a(x)$ solved along the path from the cosmology: the expansion inside the box shifts the line centres by up to $\sim10^{-3}$ of the box velocity |
| **detuning** | $\nu = -(c/\lambda)\,\Delta v/c$ (first order) | a Taylor expansion of the velocity–wavelength map | the exact exponential map $\nu = (c/\lambda)\,\mathrm{expm1}(-\Delta v/c)$; the difference is $(\Delta v/c)/2$ relative, only in the far wings |
| **pixel average** | a cell's profile averaged over the pixel when $b < 2\,dv$ | the pixel-integrated flux is not the flux at the pixel centre | SkewerJAX evaluates $F$ at the pixel centres (a convention, not an error): the average is available as `pixel_offsets` / `pixel_weights` |
| **tautail** | stops the profile where $\tau < 10^{-7}$ per cell | truncated damping wings | full Voigt profiles: the wings of strong absorbers are kept |
| **mass constant** | $n = \rho X/(A m_p)$ | the proton mass is used as the atomic mass unit | $m_u$: $A$ is an atomic weight in units of $m_u$ ($m_p/m_u = 1.00728$, +0.73 % in every density) |
| **T and b constants** | its own $k_B$ and $m_p$ | rounded constants | CODATA 2018 |
| **neutral fraction** | `sf_neutral`: a density threshold in $\rho/m_p$ | its own star-formation rule and UVB table | TNG's own SFR > 0 flag and its own UV background (TREECOOL); cold-phase $x_{\rm HI}$ only where the simulation formed stars |
| **CLOUDY density / interpolation** | hden $= 0.774\,\rho/m_p$; cubic spline in $\log f$ | not the hydrogen density; the spline overshoots near the $-30$ floor of the table | hden $= n_{\rm H}$, CLOUDY's own coordinate; linear interpolation in $\log f$ is monotonic |

The table below ($\Delta\langle F\rangle$, max $\vert\Delta F\vert$ and the P1D change of every switch, along the chain and alone) ranks them.

### Which differences matter? The effect chain

The chain switches fake_spectra's options to SkewerJAX's one at a time (integration on fake_spectra's fields, then
the extraction); "alone" applies each switch by itself. The chain telescopes exactly, so the bars add up to the
total difference.

In [ ]:
rows = effect_rows(res_i, dv)[0] + effect_rows(res_e, dv)[0]
fig, ax = plt.subplots(figsize=(15, 7), layout="constrained")
y = np.arange(len(rows))
ax.barh(y + 0.2, [r["dF_chain"] for r in rows], 0.4, color=TOL["blue"], label="along the chain")
ax.barh(y - 0.2, [r["dF_iso"] for r in rows], 0.4, color=TOL["yellow"], label="alone")
ax.set_yticks(y, [r["effect"] for r in rows]); ax.axvline(0, color=FG, lw=1)
ax.set_xscale("symlog", linthresh=1e-6); ax.set_xlabel(r"$\Delta\langle F\rangle$ of each switch (symmetric log scale)"); ax.legend()
fig.suptitle(f"Mean flux from fake_spectra to SkewerJAX: {sum(r['dF_chain'] for r in rows):+.4f} in total", fontsize=21); plt.show()

The same chain as a table: $\Delta\langle F\rangle$ and the max $\vert\Delta F\vert$ of each switch along the chain and alone, and the change of
the P1D in two $k$ bands (s/km).

In [ ]:
from skewerjax.comparison import print_effect_rows
rows_i, tot_i = effect_rows(res_i, dv)
rows_e, tot_e = effect_rows(res_e, dv)
print_effect_rows(rows_i, tot_i, "integration (on fake_spectra's extraction)")
print_effect_rows(rows_e, tot_e, "extraction (on SkewerJAX's integration)")

Each switch is an isolated option (`ExtractionOptions`, `IntegrationOptions`, `comparison.MatchedIntegration`), so any
combination can be evaluated with `matched_tau(sk, ion, lines, v, ext, integ)`. To use the comparison without
fake_spectra installed, export a reference file once in live mode (`FS.export("fake_spectra_reference.h5")`, or
`check_fake_spectra.py --export FILE`) and pass `FakeSpectraSource(reference=...)`.

## Summary

- **Agreement.** In matched mode SkewerJAX reproduces fake_spectra: max $\vert\Delta F\vert = 1.2\times10^{-5}$ and rms
  $1.7\times10^{-6}$ on the skewer above. On the benchmark skewers of notebook 10 (TNG50-4 and TNG50-3, up to 64 stored
  skewers per point), the rms of $\Delta F$ stays below $5\times10^{-6}$ for $\mathrm{Ly}\alpha$ and Si III. The max
  $\vert\Delta F\vert$ is $\leq 3\times10^{-5}$ ($\mathrm{Ly}\alpha$) and $\leq 6\times10^{-5}$ (Si III) up to 4 skewers.
  It reaches $2\times10^{-4}$ ($\mathrm{Ly}\alpha$) and $3.5\times10^{-4}$ (Si III) once the stored set holds one
  outlying pixel of one skewer per line. So every difference of the default is one of the named switches, and
  nothing else.
- **What dominates.** The **7/8 normalisation** is the largest shift of the mean flux ($\Delta\langle F\rangle = -0.026$
  here): every $\tau$ of fake_spectra is 12.5 % low before saturation, a quadrature artefact of a top-hat kernel
  sampled with SPH weights. The **deposition** changes the spectra most (where the gas sits, and its $T$ and $v$): it
  moves column between pixels and changes the P1D most at $0.1 < k < 1$ s/km. The **frame** hardly changes
  $\langle F\rangle$ but shifts features (the largest max $\vert\Delta F\vert$ of the integration switches, the P1D at
  high $k$).
- **What is small.** The chord quadrature, the mass and $b$ constants ($10^{-3}$ in $\langle F\rangle$), the linear
  detuning (first order in $\Delta v/c$), the pixel average (narrow metal lines only, $b < 2\,dv$) and `tautail` (far
  wings only).
- **Why SkewerJAX is the more accurate.** It integrates the simulated fluid as it was evolved (Voronoi cells, exact
  piece lengths, so the column is exact for any sampling), with $a(x)$ along the path and the exact velocity mapping,
  full Voigt profiles and CODATA constants. Each fake_spectra choice stays available as an isolated option, so any
  combination can be evaluated (`matched_tau(sk, ion, lines, v, ext, integ)`).
- **Open.**
  - The single outlying pixels behind the max $\vert\Delta F\vert$ at $N \geq 16$ (one $\mathrm{Ly}\alpha$ and one
    Si III skewer of the benchmark set) have not been investigated.
  - TNG50-2 is not in the new benchmark's agreement yet: its SkewerJAX points are left for the full preset. An
    earlier run on 4 skewers found Si III at $1.2\times10^{-3}$, above the other snapshots.
